Это бот для выгрузку файлов по очередям с транскрибайией, выгрузка идет в чат выгрузки отчетов по очередям и при выполнении одной из команд работает скрипт

In [1]:
import pandas as pd
from datetime import datetime
from aiogram import Bot, Dispatcher, types
from aiogram import executor

from pathlib import Path
import asyncio
import nest_asyncio
import pymysql
from aiogram.types import InlineKeyboardMarkup, InlineKeyboardButton
from aiogram.utils import executor
import asyncio

token = '6713243699:AAGXEqzNpLFeQm2-LnKeRx7yOMJnp_oOe3I'

bot = Bot(token=token)
dp = Dispatcher(bot)

In [2]:
@dp.message_handler(commands=['otchet'])
async def otchet(message):
    chat_id = str(message.chat.id)
    await bot.send_message(chat_id, "Скрипт по отчету очередей 9083, 9084, 9085")
    
    import pandas as pd
    import pymysql
    import gspread 
    from oauth2client.service_account import ServiceAccountCredentials
    import datetime
    import os
    from datetime import datetime, timedelta
    import glob
    from datetime import datetime
    from clickhouse_driver import Client
    import re
    
    
    
    path_to_credential = r'C:\Users\Guest\Downloads\вспомогательные файлы\quotas-338711-1e6d339f9a93.json' 

    scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

    credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)
    gs = gspread.authorize(credentials)

    table_name4 = 'Номера для отчета по очередям 9083-9084-9085'

    work_sheet4 = gs.open(table_name4)
    sheet4 = work_sheet4.worksheet('9083-9085')
    data4 = sheet4.get_all_values() 
    headers4 = data4.pop(0) 
    tab = pd.DataFrame(data4, columns=headers4)
    path_to_credential = r'C:\Users\Guest\Downloads\вспомогательные файлы\quotas-338711-1e6d339f9a93.json' 

    scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

    credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)
    gs = gspread.authorize(credentials)

    table_name4 = 'Номера для отчета по очередям 9083-9084-9085'

    work_sheet4 = gs.open(table_name4)
    sheet4 = work_sheet4.worksheet('Шаги')
    data4 = sheet4.get_all_values() 
    headers4 = data4.pop(0) 
    steps = pd.DataFrame(data4, columns=headers4)

    
    tab = tab.astype('str')
    tab['MSISDN'] = tab['MSISDN'].apply(lambda x: '8' + x if x.startswith('9') else x)
    import pandas as pd
    import pymysql.cursors
    import glob

    queue_list = [9083, 9084, 9085]

    user='robot_read_only'
    password='du9Itg5bnzTb'
# servers = ['192.168.1.15',
#            '192.168.1.81'#
#            ,'192.168.1.36'
#            ,'192.168.1.84'
#            ,'192.168.1.85'#,'84.201.130.36',
           
#            ,'192.168.1.123','192.168.1.103','192.168.1.86',
#            '192.168.1.122','192.168.1.124','192.168.1.125',
#            '192.168.1.127','192.168.1.129',#'192.168.1.126',
#            '192.168.1.130','192.168.1.87','192.168.1.59']



    servers = ['192.168.1.15',
# '192.168.1.131',
'192.168.1.81',
'192.168.1.36',
'192.168.1.84',
'192.168.1.85',
# '192.168.1.123',
'192.168.1.103',
'192.168.1.86',
'192.168.1.122',
'192.168.1.124',
'192.168.1.125',
'192.168.1.127',
'192.168.1.129',
'192.168.1.126',
'192.168.1.130',
'192.168.1.87',
'192.168.1.59',
'192.168.1.114',
'192.168.1.146',
'192.168.1.109',
'192.168.1.107',
'192.168.1.156',
'192.168.1.110',
'192.168.1.113',
'192.168.1.118',
'192.168.1.116',
'192.168.1.147']
              


    no_data_rollback = pd.DataFrame()
    k = 0

    df_full = pd.DataFrame()

    for queue in queue_list:
        for server in servers:
            try:
                connection = pymysql.connect(host= server,
                                 user=user,
                                 password=password,
                                 database='robot',
                                 cursorclass=pymysql.cursors.DictCursor)
                with connection.cursor() as cursor:
        
                    sql = f'''select distinct phone,date,step_start,a.uniqueid uniq_cdr,normalized,full_normalized,`set`,`match`,result,{queue} as dialog
                 from `cel_just-call-{queue}-new_1`  a
                 left join `bill_cdr` b  on a.uniqueid = b.uniqueid
                 where # step_start in (3,4,5,6,7,260,261,262,92)
                 # and 
                 #(date(date) = DATE(now()) or
                 (date(date) = DATE(now())
                 or
                 date(date) = DATE(now())- interval 1 day)
                 '''
                    cursor.execute(sql)
                    result = cursor.fetchone()
                    df = pd.read_sql_query(sql, connection)
    
                    df_full = df_full.append(df)
            except Exception as e:
                print(f"Ошибка при подключении к серверу {server}: {e}")
                continue
                k += 1
        
        
    df_full2 = df_full[['phone','step_start','uniq_cdr','full_normalized']]
    df_full2['full_normalized'] = df_full2['full_normalized'].fillna('').apply(lambda x:  'тишина' if x == '' else x)
    df_full2['route'] = df_full2['step_start'].astype('str')+', абонент: '+df_full2['full_normalized'].fillna('')
    df_full2 = df_full2[['phone','route']].drop_duplicates()
    df_full2 = df_full2.groupby('phone')['route'].apply(lambda x: ', '.join(x)).reset_index()

    import pymysql

    billsec_full = pd.DataFrame()

    for queue in queue_list:
        sql = f'''select phone, real_billsec, billsec, last_step, call_date + interval 3 hour call_date,uniqueid,
    {queue} dialog, if(stoplist_c like '%^s^%','СтопЛист','') stoplist, uniqueid uniq_billsec
from suitecrm_robot.jc_robot_log
         left join suitecrm.contacts
                   on phone = phone_work
         left join suitecrm.contacts_cstm on contacts.id = id_c
where substring(dialog, 11, 4) = {queue}
  and (date(call_date) = date(now())
    or date(call_date) = date(now()) - interval 1 day)
    '''

        Con = pymysql.Connect(host="192.168.1.182", user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
                      charset='utf8')

        billsec = pd.read_sql_query(sql, Con)
        billsec_full = billsec_full.append(billsec)
        k += 1
        
    try:    
        billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('nan',''))
        billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('nan',''))    
        
        billsec_full = billsec_full.astype('str').fillna('')
        df_full2 = df_full2.astype('str')
        def check_bill(row):
            if pd.isnull(row['real_billsec']) or row['real_billsec'] == '' or row['real_billsec'] == 'None':
                return row['billsec']
            else:
                return row['real_billsec']
        
        billsec_full['real_billsec'] = billsec_full.apply(check_bill, axis=1)
        tab = tab.astype('str')

        tab['MSISDN'] = tab['MSISDN'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['phone'] = billsec_full['phone'].astype('str').apply(lambda x: x.replace('.0',''))
        df_full2['phone'] = df_full2['phone'].astype('str').apply(lambda x: x.replace('.0',''))
        steps['last_step'] = steps['last_step'].astype('str').apply(lambda x: x.replace('.0',''))

        
    
        tab = tab.merge(df_full2, left_on = 'MSISDN', right_on = 'phone', how = 'left').fillna('')
        tab = tab.merge(billsec_full, left_on = ['MSISDN',''], right_on =['phone','dialog'], how = 'left').fillna('')
        tab['last_step'] = tab['last_step'].astype('str').apply(lambda x: x.replace('.0',''))
        tab = tab.merge(steps, left_on = 'last_step', right_on = 'last_step', how = 'left').fillna('')

        tab = tab[['','PROJECT_NAME','BASE_DATE',
           'MSISDN','call_date','report_status',
           'real_billsec','route','stoplist']].rename(columns={'call_date': 'start_time',
                         'real_billsec': 'duration',
                         'route': 'full_text'})

        tab['start_time'] = pd.to_datetime(tab['start_time'])
        df=tab.copy()
        df=df.drop_duplicates().fillna('')
        df['Rank'] = df.groupby(['MSISDN',''])['start_time'].rank(ascending=False).fillna('0')

        df['Rank'] = df['Rank'].astype('str').apply(lambda x: x.replace('.0',''))
        df = df[(df['Rank'] == '1') | (df['Rank'] == '0') | (df['Rank'] == '1.5')]
        df['Rank2'] = df.groupby('MSISDN')['full_text'].rank(ascending=False).fillna('0')
        df['Rank2'] = df['Rank2'].astype('str').apply(lambda x: x.replace('.0',''))
        df['Rank3'] = df.groupby('MSISDN')['duration'].rank(ascending=False).fillna('0')
        df['Rank3'] = df['Rank3'].astype('str').apply(lambda x: x.replace('.0',''))
        df = df[(df['Rank'] == '1') | (df['Rank'] == '0') | 
                ((df['Rank'] == '1.5') & ((df['Rank2'] == '1'))) | ((df['Rank2'] == '1.5') & ((df['Rank3'] == '1')))]
        tab=df.astype('str')
        def update_step(row):
            if row['report_status'] == 'Недозвон':
                row['start_time'] = row['BASE_DATE']
                row['duration'] = '0'
#         elif row['stoplist'] =='СтопЛист':
#             row['report_status'] = 'СтопЛист'
            else:
                return row
        tab.apply(lambda row: update_step(row), axis=1)
    
        tab['MSISDN'] = tab['MSISDN'].str.replace('^8', '', regex=True)

        tab = tab.drop_duplicates()
        tab['duration'] = tab['duration'].astype('str').apply(lambda x: x.replace('None',''))
        tab = tab[['','PROJECT_NAME','BASE_DATE','MSISDN','start_time','report_status','duration','full_text','stoplist']]
        tab.to_excel('Отчет по файлу.xlsx', index=False)
    
    # тут пишешь все свои действия по выгрузке, и потом отправляешь файл
        with open('Отчет по файлу.xlsx', 'rb') as file:
            await bot.send_document(chat_id, file)
    except Exception as e:
        print(f"Ошибка: {e}")
    

In [3]:
@dp.message_handler(commands=['otchet_9159'])
async def otchet(message):
    chat_id = str(message.chat.id)
    await bot.send_message(chat_id, "Скрипт по отчету очереди 9159")
    
    import pandas as pd
    import pymysql
    import gspread 
    from oauth2client.service_account import ServiceAccountCredentials
    import datetime
    import os
    from datetime import datetime, timedelta
    import glob
    from datetime import datetime
    from clickhouse_driver import Client
    import re
    
    
    try:
        path_to_credential = r'C:\Users\Guest\Downloads\вспомогательные файлы\quotas-338711-1e6d339f9a93.json' 

        scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

        credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)
        gs = gspread.authorize(credentials)

        table_name4 = 'Номера для отчета по очередям 9083-9084-9085'

        work_sheet4 = gs.open(table_name4)
        sheet4 = work_sheet4.worksheet('9159')
        data4 = sheet4.get_all_values() 
        headers4 = data4.pop(0) 
        tab = pd.DataFrame(data4, columns=headers4)
        path_to_credential = r'C:\Users\Guest\Downloads\вспомогательные файлы\quotas-338711-1e6d339f9a93.json' 

        scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

        credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)
        gs = gspread.authorize(credentials)
    
        table_name4 = 'Номера для отчета по очередям 9083-9084-9085'

        work_sheet4 = gs.open(table_name4)
        sheet4 = work_sheet4.worksheet('Шаги 9159')
        data4 = sheet4.get_all_values() 
        headers4 = data4.pop(0) 
        steps = pd.DataFrame(data4, columns=headers4)
    

        tab = tab.astype('str')
        tab['Телефон'] = tab['Телефон'].str.replace('^7', '8', regex=True)
        tab2 = tab.copy()

        import pandas as pd
        import pymysql.cursors
        import glob

        queue_list = [9159]

        user='robot_read_only'
        password='du9Itg5bnzTb'
# servers = ['192.168.1.15',
#            '192.168.1.81'#
#            ,'192.168.1.36'
#            ,'192.168.1.84'
#            ,'192.168.1.85'#,'84.201.130.36',
           
#            ,'192.168.1.123','192.168.1.103','192.168.1.86',
#            '192.168.1.122','192.168.1.124','192.168.1.125',
#            '192.168.1.127','192.168.1.129',#'192.168.1.126',
#            '192.168.1.130','192.168.1.87','192.168.1.59']



        servers = ['192.168.1.15',
# '192.168.1.131',
'192.168.1.81',
'192.168.1.36',
'192.168.1.84',
'192.168.1.85',
# '192.168.1.123',
'192.168.1.103',
'192.168.1.86',
'192.168.1.122',
'192.168.1.124',
'192.168.1.125',
'192.168.1.127',
'192.168.1.129',
'192.168.1.126',
'192.168.1.130',
'192.168.1.87',
'192.168.1.59',
'192.168.1.114',
'192.168.1.146',
'192.168.1.109',
'192.168.1.107',
'192.168.1.156',
'192.168.1.110',
'192.168.1.113',
'192.168.1.118',
'192.168.1.116',
'192.168.1.147']


        no_data_rollback = pd.DataFrame()
        k = 0

        df_full = pd.DataFrame()
        for queue in queue_list:
            for server in servers:
                connection = pymysql.connect(host= server,
                                 user=user,
                                 password=password,
                                 database='robot',
                                 cursorclass=pymysql.cursors.DictCursor)
                with connection.cursor() as cursor:
        
                    sql = f'''select distinct phone,date,step_start,a.uniqueid,normalized,full_normalized,`set`,
                `match`,result,{queue} as dialog, type
                 from `cel_just-call-{queue}-new_1`  a
                 left join `bill_cdr` b  on a.uniqueid = b.uniqueid
                 where # step_start in (3,4,5,6,7,260,261,262,92)
                 # and 
                 date(date) >= DATE_SUB(CURDATE(), INTERVAL 3 DAY)
                 #(date(date) = DATE(now()) or
                 #date(date) = DATE(now())- interval 1 day)
                 '''
                    cursor.execute(sql)
                    result = cursor.fetchone()
                    df = pd.read_sql_query(sql, connection)
        
                    df_full = df_full.append(df)
    
                    k += 1
        
        
        df_full2 = df_full[['phone','step_start','uniqueid','full_normalized','type']]
        df_full2['full_normalized'] = df_full2['full_normalized'].fillna('').apply(lambda x:  'тишина' if x == '' else x)
        df_full3 = df_full2.copy().astype('str')

        df_full2['route'] = df_full2['step_start'].astype('str')+', абонент: '+df_full2['full_normalized'].fillna('')
        df_full2 = df_full2[['phone','uniqueid','route']].drop_duplicates()
        df_full2 = df_full2.groupby(['phone', 'uniqueid'])['route'].apply(lambda x: ', '.join(x)).reset_index()


        df_full3 = df_full3.astype('str')
        df_full3 = df_full3[df_full3['type'] == 'expect']
        df_full3 = df_full3[['phone','step_start','full_normalized']]


        df_full3[['ФИО (из диалога)','Количество в ассортименте (из диалога)','Постоянная продажа',
          'Регионы доставки товаров (из диалога)','Категория (из диалога)', 'Комментарий',
          'Количество в наличии (из диалога)','Регионы оказания услуг (из диалога)','Время для звонка']] = ''

        def update_step(row):
            if row['step_start'] == '6':
                row['ФИО (из диалога)'] = row['full_normalized']
            elif row['step_start'] == '14':
                row['Количество в ассортименте (из диалога)'] = row['full_normalized']
            elif row['step_start'] == '16':
                row['Комментарий'] = 'Услуги'
                row['Категория (из диалога)'] = row['full_normalized']
            elif row['step_start'] == '11':
                row['Комментарий'] = 'Товары'
                row['Категория (из диалога)'] = row['full_normalized']
            elif row['step_start'] == '12':
                row['Постоянная продажа'] = row['full_normalized']
            elif row['step_start'] == '13':
                row['Регионы доставки товаров (из диалога)'] = row['full_normalized'] 
            elif row['step_start'] == '15':
                row['Количество в наличии (из диалога)'] = row['full_normalized']
            elif row['step_start'] == '18':
                row['Регионы оказания услуг (из диалога)'] = row['full_normalized']
            elif row['step_start'] in ['233', '237', '238', '239', '240', '241', '242', '243', '235', '236', '234','90']:
                row['Время для звонка'] = row['full_normalized']
            else:
                return row
    
        df_full3.apply(lambda row: update_step(row), axis=1)
        df_full3 = df_full3[['phone','ФИО (из диалога)','Количество в ассортименте (из диалога)','Постоянная продажа',
              'Регионы доставки товаров (из диалога)','Категория (из диалога)', 'Комментарий',
          'Количество в наличии (из диалога)','Регионы оказания услуг (из диалога)','Время для звонка']]

        df_full3 = df_full3.loc[(df_full3['ФИО (из диалога)'] != '') | (df_full3['Количество в ассортименте (из диалога)'] != '') | 
                        (df_full3['Постоянная продажа'] != '') |(df_full3['Регионы доставки товаров (из диалога)'] != '') |
                        (df_full3['Категория (из диалога)'] != '') |(df_full3['Комментарий'] != '') |
                        (df_full3['Количество в наличии (из диалога)'] != '') |(df_full3['Регионы оказания услуг (из диалога)'] != '') |
                       (df_full3['Время для звонка'] != '') ]

        import pymysql

        billsec_full = pd.DataFrame()

        for queue in queue_list:
            sql = f'''select phone, real_billsec,billsec, server_number, last_step, call_date + interval 3 hour call_date,uniqueid
    from suitecrm_robot.jc_robot_log
    where substring(dialog,11,4) = {queue} and
    (date(call_date) >= DATE_SUB(CURDATE(), INTERVAL 3 DAY))
    #(date(call_date) = date(now())
    #   or date(call_date) = date(now())- interval 1 day)
    '''

            Con = pymysql.Connect(host="192.168.1.182", user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
                      charset='utf8')

            billsec = pd.read_sql_query(sql, Con)
            billsec_full = billsec_full.append(billsec)
            k += 1

        
        billsec_full = billsec_full.astype('str').fillna('')
        df_full3 = df_full3.groupby('phone').agg(' '.join).fillna('')
        df_full3.reset_index(inplace=True)
        billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('nan',''))
        billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('nan',''))
        df_full2 = df_full2.astype('str')
        def check_bill(row):
            if pd.isnull(row['real_billsec']) or row['real_billsec'] == '' or row['real_billsec'] == 'None':
                return row['billsec']
            else:
                return row['real_billsec']
        billsec_full['real_billsec'] = billsec_full.apply(check_bill, axis=1)
        tab['Телефон'] = tab['Телефон'].astype('str').apply(lambda x: x.replace('.0',''))
        df_full2['phone'] = df_full2['phone'].astype('str').apply(lambda x: x.replace('.0',''))
        df_full3['phone'] = df_full3['phone'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['phone'] = billsec_full['phone'].astype('str').apply(lambda x: x.replace('.0',''))
        tab = tab.merge(df_full2, left_on = 'Телефон', right_on = 'phone', how = 'left').fillna('')
        tab = tab.merge(df_full3, left_on = 'Телефон', right_on = 'phone', how = 'left').fillna('')
        tab = tab.merge(billsec_full, left_on = 'Телефон', right_on = 'phone', how = 'left').fillna('')
        
        steps['last_step'] = steps['last_step'].astype('str').apply(lambda x: x.replace('.0',''))
        tab = tab.merge(steps, left_on = 'last_step', right_on = 'last_step', how = 'left').fillna('')

        tab = tab.drop_duplicates()
        tab2 = tab[tab['phone_x'] != '']
        tab1 = tab2[['call_date',
             'real_billsec',
             'ИНН',
             'Название компании',
             'ФИО','ФИО (из диалога)',
             'Телефон','Город',
             'Регион','Комментарий',
             'Постоянная продажа',
             'Рубрика','Подрубрика',
             'Категория (из диалога)',
             'Количество в ассортименте (из диалога)',
             'Количество в наличии (из диалога)',
             'Регионы доставки товаров (из диалога)',
             'Регионы оказания услуг (из диалога)','Время для звонка',
             'report_status','route']].rename(columns={'call_date': 'Время звонка',
                         'real_billsec': 'Продолжительность звонка (sec)',
                         'report_status': 'Статус',
                         'route': 'Текст диалога'})
        tab1.to_excel('Отчет по файлу 9159.xlsx', index=False)

    


    # тут пишешь все свои действия по выгрузке, и потом отправляешь файл
        with open('Отчет по файлу 9159.xlsx', 'rb') as file:
            await bot.send_document(chat_id, file)
    except Exception as e:
        print(f"Ошибка: {e}")
    

In [4]:
@dp.message_handler(commands=['otchet_9046'])
async def otchet(message):
    chat_id = str(message.chat.id)
    await bot.send_message(chat_id, "Скрипт по отчету очереди 9046")
    
    import pandas as pd
    import pymysql
    import gspread 
    from oauth2client.service_account import ServiceAccountCredentials
    import datetime
    import os
    from datetime import datetime, timedelta
    import glob
    from datetime import datetime
    from clickhouse_driver import Client
    import re
    

    path_to_credential = r'C:\Users\Guest\Downloads\вспомогательные файлы\quotas-338711-1e6d339f9a93.json' 

    scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

    credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)
    gs = gspread.authorize(credentials)

    table_name4 = 'Номера для отчета по очередям 9083-9084-9085'

    work_sheet4 = gs.open(table_name4)
    sheet4 = work_sheet4.worksheet('Шаги 9046')
    data4 = sheet4.get_all_values() 
    headers4 = data4.pop(0) 
    steps = pd.DataFrame(data4, columns=headers4)

    import pandas as pd
    import pymysql.cursors
    import glob

    queue_list = [9046]

    user='robot_read_only'
    password='du9Itg5bnzTb'
# servers = ['192.168.1.15',
#            '192.168.1.81'#
#            ,'192.168.1.36'
#            ,'192.168.1.84'
#            ,'192.168.1.85'#,'84.201.130.36',
           
#            ,'192.168.1.123','192.168.1.103','192.168.1.86',
#            '192.168.1.122','192.168.1.124','192.168.1.125',
#            '192.168.1.127','192.168.1.129',#'192.168.1.126',
#            '192.168.1.130','192.168.1.87','192.168.1.59']



    servers = ['192.168.1.15',
# '192.168.1.131',
'192.168.1.81',
'192.168.1.36',
'192.168.1.84',
'192.168.1.85',
# '192.168.1.123',
'192.168.1.103',
'192.168.1.86',
'192.168.1.122',
'192.168.1.124',
'192.168.1.125',
'192.168.1.127',
'192.168.1.129',
'192.168.1.126',
'192.168.1.130',
'192.168.1.87',
'192.168.1.59',
'192.168.1.114',
'192.168.1.146',
'192.168.1.109',
'192.168.1.107',
'192.168.1.156',
'192.168.1.110',
'192.168.1.113',
'192.168.1.118',
'192.168.1.116',
'192.168.1.147']
              


    no_data_rollback = pd.DataFrame()
    k = 0

    df_full = pd.DataFrame()

    for queue in queue_list:
        for server in servers:
            try:
                connection = pymysql.connect(host= server,
                                 user=user,
                                 password=password,
                                 database='robot',
                                 cursorclass=pymysql.cursors.DictCursor)
                with connection.cursor() as cursor:
        
                    sql = f'''select distinct phone,date,step_start,a.uniqueid,normalized,full_normalized,`set`,`match`,result,{queue} as dialog
                 from `cel_just-call-{queue}-new_1`  a
                 left join `bill_cdr` b  on a.uniqueid = b.uniqueid
                 where # step_start in (3,4,5,6,7,260,261,262,92)
                 # and 
                 #(date(date) = DATE(now()) or
                 (date(date) = DATE(now())
                 or
                 date(date) = DATE(now())- interval 1 day)
                 '''
                    cursor.execute(sql)
                    result = cursor.fetchone()
                    df = pd.read_sql_query(sql, connection)
    
                    df_full = df_full.append(df)
            except Exception as e:
                print(f"Ошибка при подключении к серверу {server}: {e}")
                continue
                k += 1
        
        
    df_full2 = df_full[['phone','step_start','uniqueid','full_normalized']]
    df_full2['full_normalized'] = df_full2['full_normalized'].fillna('').apply(lambda x:  'тишина' if x == '' else x)
    df_full2['route'] = df_full2['step_start'].astype('str')+', абонент: '+df_full2['full_normalized'].fillna('')
    df_full2 = df_full2[['phone','uniqueid','route']].drop_duplicates()
    df_full2 = df_full2.groupby(['phone', 'uniqueid'])['route'].apply(lambda x: ', '.join(x)).reset_index()

    import pymysql

    billsec_full = pd.DataFrame()

    for queue in queue_list:
        sql = f'''select phone, real_billsec, billsec, last_step, call_date + interval 3 hour call_date,{queue}  dialog,uniqueid
    from suitecrm_robot.jc_robot_log
    where substring(dialog,11,4) = {queue} and
    (date(call_date) = date(now())
       or date(call_date) = date(now())- interval 1 day)
    '''

        Con = pymysql.Connect(host="192.168.1.182", user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
                      charset='utf8')

        billsec = pd.read_sql_query(sql, Con)
        billsec_full = billsec_full.append(billsec)
        k += 1
        
        
    billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('.0',''))
    billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('nan',''))
    billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('.0',''))
    billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('nan',''))    
        
    billsec_full = billsec_full.astype('str').fillna('')
    df_full2 = df_full2.astype('str')
    def check_bill(row):
        if pd.isnull(row['real_billsec']) or row['real_billsec'] == '' or row['real_billsec'] == 'None':
            return row['billsec']
        else:
            return row['real_billsec']
        
    billsec_full['real_billsec'] = billsec_full.apply(check_bill, axis=1)

    billsec_full['phone'] = billsec_full['phone'].astype('str').apply(lambda x: x.replace('.0',''))
    df_full2['phone'] = df_full2['phone'].astype('str').apply(lambda x: x.replace('.0',''))
    steps['last_step'] = steps['last_step'].astype('str').apply(lambda x: x.replace('.0',''))
    
    tab = billsec_full.merge(df_full2, left_on = 'phone', right_on = 'phone', how = 'left').fillna('')
    tab = tab.merge(steps, left_on = 'last_step', right_on = 'last_step', how = 'left').fillna('')

    tab.to_excel('Отчет 9046.xlsx', index=False)
    
    # тут пишешь все свои действия по выгрузке, и потом отправляешь файл
    with open('Отчет 9046.xlsx', 'rb') as file:
        await bot.send_document(chat_id, file)
    

In [5]:
@dp.message_handler(commands=['otchet_9166'])
async def otchet(message):
    chat_id = str(message.chat.id)
    await bot.send_message(chat_id, "Скрипт по отчету очереди 9166")
    
    import pandas as pd
    import pymysql
    import gspread 
    from oauth2client.service_account import ServiceAccountCredentials
    import datetime
    import os
    from datetime import datetime, timedelta
    import glob
    from datetime import datetime
    from clickhouse_driver import Client
    import re
    

    path_to_credential = r'C:\Users\Guest\Downloads\вспомогательные файлы\quotas-338711-1e6d339f9a93.json' 

    scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

    credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)
    gs = gspread.authorize(credentials)

    table_name4 = 'Номера для отчета по очередям 9083-9084-9085'

    work_sheet4 = gs.open(table_name4)
    sheet4 = work_sheet4.worksheet('9166')
    data4 = sheet4.get_all_values() 
    headers4 = data4.pop(0) 
    tab = pd.DataFrame(data4, columns=headers4)

    tab = tab.astype('str')


    path_to_credential = r'C:\Users\Guest\Downloads\вспомогательные файлы\quotas-338711-1e6d339f9a93.json' 

    scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

    credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)
    gs = gspread.authorize(credentials)

    table_name4 = 'Номера для отчета по очередям 9083-9084-9085'

    work_sheet4 = gs.open(table_name4)
    sheet4 = work_sheet4.worksheet('Шаги 9166')
    data4 = sheet4.get_all_values() 
    headers4 = data4.pop(0) 
    steps = pd.DataFrame(data4, columns=headers4)

    import pandas as pd
    import pymysql.cursors
    import glob

    queue_list = [9166]

    user='robot_read_only'
    password='du9Itg5bnzTb'
# servers = ['192.168.1.15',
#            '192.168.1.81'#
#            ,'192.168.1.36'
#            ,'192.168.1.84'
#            ,'192.168.1.85'#,'84.201.130.36',
           
#            ,'192.168.1.123','192.168.1.103','192.168.1.86',
#            '192.168.1.122','192.168.1.124','192.168.1.125',
#            '192.168.1.127','192.168.1.129',#'192.168.1.126',
#            '192.168.1.130','192.168.1.87','192.168.1.59']



    servers = ['192.168.1.15',
# '192.168.1.131',
'192.168.1.81',
'192.168.1.36',
'192.168.1.84',
'192.168.1.85',
# '192.168.1.123',
'192.168.1.103',
'192.168.1.86',
'192.168.1.122',
'192.168.1.124',
'192.168.1.125',
'192.168.1.127',
'192.168.1.129',
'192.168.1.126',
'192.168.1.130',
'192.168.1.87',
'192.168.1.59',
'192.168.1.114',
'192.168.1.146',
'192.168.1.109',
'192.168.1.107',
'192.168.1.156',
'192.168.1.110',
'192.168.1.113',
'192.168.1.118',
'192.168.1.116',
'192.168.1.147']
              


    no_data_rollback = pd.DataFrame()
    k = 0

    df_full = pd.DataFrame()

    for queue in queue_list:
        for server in servers:
            try:
                connection = pymysql.connect(host= server,
                                 user=user,
                                 password=password,
                                 database='robot',
                                 cursorclass=pymysql.cursors.DictCursor)
                with connection.cursor() as cursor:
        
                    sql = f'''select distinct phone,date,step_start,normalized,full_normalized,`set`,`match`,result,{queue} as dialog
                 from `cel_just-call-{queue}-new_1`  a
                 left join `bill_cdr` b  on a.uniqueid = b.uniqueid
                 where # step_start in (3,4,5,6,7,260,261,262,92)
                 # and 
                 #(date(date) = DATE(now()) or
                 (date(date) = DATE(now())
                 or
                 date(date) = DATE(now())- interval 1 day)
                 '''
                    cursor.execute(sql)
                    result = cursor.fetchone()
                    df = pd.read_sql_query(sql, connection)
    
                    df_full = df_full.append(df)
            except Exception as e:
                print(f"Ошибка при подключении к серверу {server}: {e}")
                continue
                k += 1
        
        
    df_full2 = df_full[['phone','step_start','full_normalized']]
    df_full2['full_normalized'] = df_full2['full_normalized'].fillna('').apply(lambda x:  'тишина' if x == '' else x)
    df_full2['route'] = df_full2['step_start'].astype('str')+', абонент: '+df_full2['full_normalized'].fillna('')
    df_full2 = df_full2[['phone','route']].drop_duplicates()
    df_full2 = df_full2.groupby('phone')['route'].apply(lambda x: ', '.join(x)).reset_index()

    import pymysql

    billsec_full = pd.DataFrame()

    for queue in queue_list:
        sql = f'''select phone, real_billsec, billsec, last_step, call_date + interval 3 hour call_date,
        {queue}  dialog , was_stepgroups complited, route rt
    from suitecrm_robot.jc_robot_log
    where substring(dialog,11,4) = {queue} and
    (date(call_date) = date(now())
       or date(call_date) = date(now())- interval 1 day)
    '''

        Con = pymysql.Connect(host="192.168.1.182", user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
                      charset='utf8')

        billsec = pd.read_sql_query(sql, Con)
        billsec_full = billsec_full.append(billsec)
        k += 1
        
        
    try:
        billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('nan',''))
        billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('nan',''))    
        
        billsec_full = billsec_full.astype('str').fillna('')
        df_full2 = df_full2.astype('str')
        def check_bill(row):
            if pd.isnull(row['real_billsec']) or row['real_billsec'] == '' or row['real_billsec'] == 'None':
                return row['billsec']
            else:
                return row['real_billsec']
        
        billsec_full['real_billsec'] = billsec_full.apply(check_bill, axis=1)
        tab = tab.astype('str')

        tab['customer_phone'] = tab['customer_phone'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['phone'] = billsec_full['phone'].astype('str').apply(lambda x: x.replace('.0',''))
        df_full2['phone'] = df_full2['phone'].astype('str').apply(lambda x: x.replace('.0',''))
        steps['last_step'] = steps['last_step'].astype('str').apply(lambda x: x.replace('.0',''))


        tab = tab.merge(billsec_full, left_on = 'customer_phone', right_on ='phone', how = 'left').fillna('')    
        tab = tab.merge(df_full2, left_on = 'customer_phone', right_on = 'phone', how = 'left').fillna('')
        tab = tab.merge(steps, left_on = 'last_step', right_on = 'last_step', how = 'left').fillna('')
        tab1 = tab.copy()
        tab = tab[['customer_phone','call_date','report_status',
           'real_billsec','route','complited','rt']].rename(columns={'real_billsec': 'duration',
                         'route': 'full_text'})
        tab=tab.astype('str')
    
        def update_step(row):
            if row['report_status'] == 'Недозвон':
                row['duration'] = '0'
            else:
                return row
        tab.apply(lambda row: update_step(row), axis=1)

        tab = tab.drop_duplicates()
        tab['duration'] = tab['duration'].astype('str').apply(lambda x: x.replace('None',''))
        tab[['start_date', 'start_time']] = tab['call_date'].str.split(' ', expand=True).fillna('')
        tab[['PROJECT_NAME','BASE_DATE']] = ''
        tab = tab[['PROJECT_NAME','BASE_DATE','customer_phone','start_date','start_time','duration','report_status','full_text']]
        tab1 = tab1[['customer_phone','call_date','report_status',
           'real_billsec','route','last_step','complited','rt']].rename(columns={'call_date': 'start_time',
                         'real_billsec': 'duration',
                         'route': 'full_text'
                            })
        tab1=tab1.astype('str')
        tab1[['Успешный звонок (диалог состоялся)','Отказ отвечать',
     'Недозвон','Согласие, добавлено оператором (ботом)',
     'Согласие, отправлено SMS','Отказ от предложения',
     'Неактуально','Просит перезвонить','Агрессия','Без обработки',
      'Сомнение','Дети','Пожилой человек']]=''
        tab1['rt'] = tab1['rt'].astype('str')+','
        def update_steps(row):
           
            if row['last_step'] in ['102','103', '90', '10', '209', '409'] and '10,' in row['rt']:
                row['Отказ от предложения'] = 1
            
            elif row['last_step'] in ['111', '261', '1','262']:
                row['Недозвон'] = 1
            
#             elif row['last_step'] == '2':
#                 row['Отказ отвечать'] = 1
                     
            elif row['last_step'] == '92':
                row['Согласие, добавлено оператором (ботом)'] = 1
            
            elif row['last_step'] == '104':
                row['Согласие, отправлено SMS '] = 1
            
            elif row['last_step'] == '103':
                row['Неактуально'] = 1
            
            elif row['last_step'] == '499':
                row['Агрессия'] = 1
            
            elif row['last_step']  in ['101', '250']:
                row['Просит перезвонить'] = 1
                
            elif row['last_step']  in ['101', '250'] and '202' in  row['rt']:
                row['Сомнение'] = 1
            else:
                return row
    
        tab1.apply(lambda row: update_steps(row), axis=1).fillna('').astype('str')
        def update_step(row):
            if row['start_time'] != '':
                if row['Недозвон'] != '1' or row['Недозвон'] != 1:
                    if '10,' in row['rt'] and row['last_step'] != '111':
                        row['Успешный звонок (диалог состоялся)'] = 1 
                    elif  '10,' not in row['rt'] and row['Недозвон'] != 1:
                        row['Отказ отвечать'] = 1 
                else:
                    return row
        tab1.apply(lambda row: update_step(row), axis=1).fillna('').astype('str')
        tab1[['Дата', 'Время']] = tab1['start_time'].str.split(' ', expand=True).fillna('')
        tab1 = tab1[['start_time',
             'customer_phone',
             'Успешный звонок (диалог состоялся)',
             'Отказ отвечать',
             'Недозвон',
             'Без обработки',
             'Согласие, добавлено оператором (ботом)',
             'Согласие, отправлено SMS',
             'Сомнение',
             'Отказ от предложения',
             'Неактуально',
             'Дети',
             'Пожилой человек',
             'Просит перезвонить',
             'Агрессия','Дата', 'Время',
             'full_text','complited']].rename(columns={'start_time': 'Дата и время звонка',
                         'customer_phone': 'Номер абонента',
                         'full_text': 'Транскрибация диалога','complited': 'Вызов завершен'})
        tab.to_excel('9166(1).xlsx', index=False)
        tab1.to_excel('9166(2).xlsx', index=False)
        with open('9166(1).xlsx', 'rb') as file1, open('9166(2).xlsx', 'rb') as file2:
            await bot.send_document(chat_id, file1)
            await bot.send_document(chat_id, file2)
    except Exception as e:
        print(f"Ошибка: {e}")
        

In [6]:
@dp.message_handler(commands=['audio_9159'])
async def otchet(message):
    chat_id = str(message.chat.id)
    await bot.send_message(chat_id, "Выгружаю аудио по диалогу 9159")
    import pandas as pd
    import gspread
    from oauth2client.service_account import ServiceAccountCredentials
    import pymysql
    import datetime
    import os
    import glob
    import requests
    import wget
    import speech_recognition as sr

    import datetime
    try:
        path = r'C:\Users\Guest\Desktop\audios'

        servers = [
'192.168.1.15',
'192.168.1.131',
'192.168.1.81',
'192.168.1.36',
'192.168.1.84',
'192.168.1.85',
# '192.168.1.123',
'192.168.1.103',
'192.168.1.86',
'192.168.1.122',
'192.168.1.124',
'192.168.1.125',
'192.168.1.127',
'192.168.1.129',
'192.168.1.126',
'192.168.1.130',
'192.168.1.87',
'192.168.1.59',        
'192.168.1.114',
'192.168.1.146',           
'192.168.1.109',
'192.168.1.107',           
'192.168.1.156',           
'192.168.1.110',           
'192.168.1.113',           
'192.168.1.118',           
'192.168.1.116',           
'192.168.1.147'           
          ]
        path_to_credential = r'C:\Users\Guest\Downloads\вспомогательные файлы\quotas-338711-1e6d339f9a93.json' 
        table_name = 'Номера для отчета по очередям 9083-9084-9085'
    
        scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

        credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)
        gs = gspread.authorize(credentials)
        work_sheet = gs.open(table_name)

        sheet = work_sheet.worksheets()[3]
    
        df = pd.DataFrame(sheet.get_all_values()[1:], columns=sheet.get_all_values().pop(0))
        df['Телефон'] = df['Телефон'].str.replace('^7', '8', regex=True)
        phones = df['Телефон'].to_list()
        phones = str(phones).replace('[','').replace(']','').replace("'",'')
        sql = f'''SELECT phone, uniqueid,server_number, call_date
FROM suitecrm_robot.jc_robot_log
where phone in ({phones}) and last_step in (105,233,237,238,239,240,241,242,243,235,236,234)
and date(call_date) = date(now()) - interval 1 day
'''

        Con = pymysql.Connect(host="192.168.1.182", user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
              charset='utf8')

        robot_log = pd.read_sql_query(sql, Con)
        robot_log['id'] = robot_log['uniqueid'].apply(lambda x: x[:x.find(".",x.find(".",2)+1)])
        robot_log['new_name'] = robot_log.apply(lambda row: '9159_'+str(row['phone'])+'_'+str(row['call_date'].strftime("%d%m%Y%H%M%S"))+'.mp3', axis=1)
        dialogs = robot_log['id'].to_list()


        call_date = str(robot_log['call_date'][0].strftime("%d-%m-%Y"))
        queue = 'just-call-9159-new/'
        audios = []
        name_audios = []
        for server in servers:
            url = 'http://'+server+f':9090/{call_date}/'+queue
    
            try:
                response = requests.get(url)
                files = response.text
        
                for i in files.split('\n'):
                    if "dialog.opus" in i:

                        audio_file = i[i.find('[SND]"></td><td><a href="')+25 : i.find('dialog.opus')+11]
                        ful_url = url+'/'+audio_file
                
                        if audio_file[:audio_file.find("_")] in dialogs:
    
                            audios.append(ful_url)
                            name_audios.append(audio_file[:audio_file.find("_")])
                    
            except :
                pass
        links = pd.DataFrame({'id': name_audios, 'links': audios})
        full = robot_log.merge(links, how='left', on='id')
        full = full[['links','new_name']].fillna('').query('links != ""').values.tolist()
        for row in full:
            url = row[0]
            filename = row[1]
    
            response = requests.get(url)
            with open(path + ('\\') + filename, 'wb') as file:
                file.write(response.content)
        import shutil
        audios_dir = os.path.join(os.path.expanduser('~'), 'Desktop', 'audios')
        files = os.listdir(audios_dir)
        shutil.make_archive(os.path.join(os.path.expanduser('~'), 'Desktop', 'audios_archive'), 'zip', audios_dir)
        for file in os.listdir(path):
            file_path = os.path.join(path, file)
            try:
                if os.path.isfile(file_path):
                    os.unlink(file_path)
            except Exception as e:
                print(f"Ошибка удаления файла {file}: {e}")
        import zipfile
        
        with zipfile.ZipFile('9159.zip', 'w') as zipf:
            zipf.write(r'C:\Users\Guest\Desktop\audios_archive.zip')
        with open('9159.zip', 'rb') as file:
            await bot.send_document(chat_id, file)

    except Exception as e:
        print(f"Ошибка: {e}")
        

In [7]:
@dp.message_handler(commands=['otchet_9186'])
async def otchet(message):
    chat_id = str(message.chat.id)
    await bot.send_message(chat_id, "Скрипт по отчету очереди 9186")
    
    import pandas as pd
    import pymysql
    import gspread 
    from oauth2client.service_account import ServiceAccountCredentials
    import datetime
    import os
    from datetime import datetime, timedelta
    import glob
    from datetime import datetime
    from clickhouse_driver import Client
    import re
    
    
    
    path_to_credential = r'C:\Users\Guest\Downloads\вспомогательные файлы\quotas-338711-1e6d339f9a93.json' 

    scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

    credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)
    gs = gspread.authorize(credentials)

    table_name4 = 'Номера для отчета по очередям 9083-9084-9085'

    work_sheet4 = gs.open(table_name4)
    sheet4 = work_sheet4.worksheet('9186')
    data4 = sheet4.get_all_values() 
    headers4 = data4.pop(0) 
    tab = pd.DataFrame(data4, columns=headers4)
    path_to_credential = r'C:\Users\Guest\Downloads\вспомогательные файлы\quotas-338711-1e6d339f9a93.json' 

    scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

    credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)
    gs = gspread.authorize(credentials)

    table_name4 = 'Номера для отчета по очередям 9083-9084-9085'

    work_sheet4 = gs.open(table_name4)
    sheet4 = work_sheet4.worksheet('Шаги 9186')
    data4 = sheet4.get_all_values() 
    headers4 = data4.pop(0) 
    steps = pd.DataFrame(data4, columns=headers4)

    
    tab = tab.astype('str')
    tab['phone'] = tab['phone'].str.replace('^7', '8', regex=True)
    import pandas as pd
    import pymysql.cursors
    import glob

    queue_list = [9186]

    user='robot_read_only'
    password='du9Itg5bnzTb'
# servers = ['192.168.1.15',
#            '192.168.1.81'#
#            ,'192.168.1.36'
#            ,'192.168.1.84'
#            ,'192.168.1.85'#,'84.201.130.36',
           
#            ,'192.168.1.123','192.168.1.103','192.168.1.86',
#            '192.168.1.122','192.168.1.124','192.168.1.125',
#            '192.168.1.127','192.168.1.129',#'192.168.1.126',
#            '192.168.1.130','192.168.1.87','192.168.1.59']



    servers = ['192.168.1.15',
# '192.168.1.131',
'192.168.1.81',
'192.168.1.36',
'192.168.1.84',
'192.168.1.85',
# '192.168.1.123',
'192.168.1.103',
'192.168.1.86',
'192.168.1.122',
'192.168.1.124',
'192.168.1.125',
'192.168.1.127',
'192.168.1.129',
'192.168.1.126',
'192.168.1.130',
'192.168.1.87',
'192.168.1.59',
'192.168.1.114',
'192.168.1.146',
'192.168.1.109',
'192.168.1.107',
'192.168.1.156',
'192.168.1.110',
'192.168.1.113',
'192.168.1.118',
'192.168.1.116',
'192.168.1.147']
              


    no_data_rollback = pd.DataFrame()
    k = 0

    df_full = pd.DataFrame()

    for queue in queue_list:
        for server in servers:
            try:
                connection = pymysql.connect(host= server,
                                 user=user,
                                 password=password,
                                 database='robot',
                                 cursorclass=pymysql.cursors.DictCursor)
                with connection.cursor() as cursor:
        
                    sql = f'''select distinct phone,date,step_start,a.uniqueid uniq_cdr,normalized,full_normalized,`set`,`match`,result,{queue} as dialog
                 from `cel_just-call-{queue}-new_1`  a
                 left join `bill_cdr` b  on a.uniqueid = b.uniqueid
                 where # step_start in (3,4,5,6,7,260,261,262,92)
                 # and 
                 #(date(date) = DATE(now()) or
                 (date(date) = DATE(now())
                 or
                 date(date) = DATE(now())- interval 1 day)
                 '''
                    cursor.execute(sql)
                    result = cursor.fetchone()
                    df = pd.read_sql_query(sql, connection)
    
                    df_full = df_full.append(df)
            except Exception as e:
                print(f"Ошибка при подключении к серверу {server}: {e}")
                continue
                k += 1
        
        
    df_full2 = df_full[['phone','step_start','uniq_cdr','full_normalized']]
    df_full2['full_normalized'] = df_full2['full_normalized'].fillna('').apply(lambda x:  'тишина' if x == '' else x)
    df_full2['route'] = df_full2['step_start'].astype('str')+', абонент: '+df_full2['full_normalized'].fillna('')
    df_full2 = df_full2[['phone','route']].drop_duplicates()
    df_full2 = df_full2.groupby('phone')['route'].apply(lambda x: ', '.join(x)).reset_index()

    import pymysql

    billsec_full = pd.DataFrame()

    for queue in queue_list:
        sql = f'''select phone, real_billsec, billsec, last_step, call_date + interval 3 hour call_date,uniqueid,
    {queue} dialog, if(stoplist_c like '%^s^%','СтопЛист','') stoplist, uniqueid uniq_billsec
from suitecrm_robot.jc_robot_log
         left join suitecrm.contacts
                   on phone = phone_work
         left join suitecrm.contacts_cstm on contacts.id = id_c
where substring(dialog, 11, 4) = {queue}
  and (date(call_date) = date(now())
    or date(call_date) = date(now()) - interval 1 day)
    '''

        Con = pymysql.Connect(host="192.168.1.182", user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
                      charset='utf8')

        billsec = pd.read_sql_query(sql, Con)
        billsec_full = billsec_full.append(billsec)
        k += 1
        
    try:    
        billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('nan',''))
        billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('nan',''))    
        
        billsec_full = billsec_full.astype('str').fillna('')
        df_full2 = df_full2.astype('str')
        def check_bill(row):
            if pd.isnull(row['real_billsec']) or row['real_billsec'] == '' or row['real_billsec'] == 'None':
                return row['billsec']
            else:
                return row['real_billsec']
        
        billsec_full['real_billsec'] = billsec_full.apply(check_bill, axis=1)
        tab = tab.astype('str')

        tab['phone'] = tab['phone'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['phone'] = billsec_full['phone'].astype('str').apply(lambda x: x.replace('.0',''))
        df_full2['phone'] = df_full2['phone'].astype('str').apply(lambda x: x.replace('.0',''))
        steps['last_step'] = steps['last_step'].astype('str').apply(lambda x: x.replace('.0',''))
    
        tab = tab.merge(df_full2, left_on = 'phone', right_on = 'phone', how = 'left').fillna('')
        tab = tab.merge(billsec_full, left_on = 'phone', right_on ='phone', how = 'left').fillna('')
        tab = tab.merge(steps, left_on = 'last_step', right_on = 'last_step', how = 'left').fillna('')

        tab = tab[['phone','call_date','report_status',
           'real_billsec','route']].rename(columns={'call_date': 'start_time',
                         'real_billsec': 'duration',
                         'route': 'full_text'})

        tab['start_time'] = pd.to_datetime(tab['start_time'])
        df=tab.copy()
        df=df.drop_duplicates().fillna('')
        df['Rank'] = df.groupby('phone')['start_time'].rank(ascending=False).fillna('0')

        df['Rank'] = df['Rank'].astype('str').apply(lambda x: x.replace('.0',''))
        df = df[(df['Rank'] == '1') | (df['Rank'] == '0') | (df['Rank'] == '1.5')]
        df['Rank2'] = df.groupby('phone')['full_text'].rank(ascending=False).fillna('0')
        df['Rank2'] = df['Rank2'].astype('str').apply(lambda x: x.replace('.0',''))
        df = df[(df['Rank'] == '1') | (df['Rank'] == '0') | ((df['Rank'] == '1.5') & (df['Rank2'] == '1'))]
        tab=df.astype('str')
        def update_step(row):
            if row['report_status'] == 'Недозвон':
#                  row['start_time'] = row['BASE_DATE']
                row['duration'] = '0'
#         elif row['stoplist'] =='СтопЛист':
#             row['report_status'] = 'СтопЛист'
            else:
                return row
        tab.apply(lambda row: update_step(row), axis=1)
    
#         tab['phone'] = tab['phone'].str.replace('^8', '', regex=True)

        tab = tab.drop_duplicates()
        tab['duration'] = tab['duration'].astype('str').apply(lambda x: x.replace('None',''))
        tab['start_time'] = tab['start_time'].astype('str').apply(lambda x: x.replace('NaT',''))

        tab = tab[['phone','start_time','report_status','duration','full_text']]
        tab.to_excel('Отчет по 9186.xlsx', index=False)
    
    # тут пишешь все свои действия по выгрузке, и потом отправляешь файл
        with open('Отчет по 9186.xlsx', 'rb') as file:
            await bot.send_document(chat_id, file)
    except Exception as e:
        print(f"Ошибка: {e}")
    

In [8]:
@dp.message_handler(commands=['audio_9166'])
async def otchet(message):
    chat_id = str(message.chat.id)
    await bot.send_message(chat_id, "Выгружаю аудио за вчера по диалогу 9166")
    import pandas as pd
    import gspread
    from oauth2client.service_account import ServiceAccountCredentials
    import pymysql
    import datetime
    import os
    import glob
    import requests
    import wget
    import speech_recognition as sr
    import os
    import math
    import zipfile

    import datetime
    try:
        path = r'C:\Users\Guest\Desktop\audios'

        servers = [
'192.168.1.15',
'192.168.1.131',
'192.168.1.81',
'192.168.1.36',
'192.168.1.84',
'192.168.1.85',
# '192.168.1.123',
'192.168.1.103',
'192.168.1.86',
'192.168.1.122',
'192.168.1.124',
'192.168.1.125',
'192.168.1.127',
'192.168.1.129',
'192.168.1.126',
'192.168.1.130',
'192.168.1.87',
'192.168.1.59',        
'192.168.1.114',
'192.168.1.146',           
'192.168.1.109',
'192.168.1.107',           
'192.168.1.156',           
'192.168.1.110',           
'192.168.1.113',           
'192.168.1.118',           
'192.168.1.116',           
'192.168.1.147'           
          ]
        path_to_credential = r'C:\Users\Guest\Downloads\вспомогательные файлы\quotas-338711-1e6d339f9a93.json' 
        table_name = 'Номера для отчета по очередям 9083-9084-9085'
    
        scope = ['https://spreadsheets.google.com/feeds',
         'https://www.googleapis.com/auth/drive']

        credentials = ServiceAccountCredentials.from_json_keyfile_name(path_to_credential, scope)
        gs = gspread.authorize(credentials)
        work_sheet = gs.open(table_name)

        sheet = work_sheet.worksheets()[1]
    
        df = pd.DataFrame(sheet.get_all_values()[1:], columns=sheet.get_all_values().pop(0))

        phones = df['customer_phone'].to_list()
        phones = str(phones).replace('[','').replace(']','').replace("'",'')
        sql = f'''SELECT phone, uniqueid,server_number, call_date
        FROM suitecrm_robot.jc_robot_log
        where phone in ({phones})
        and date(call_date) = date(now()) - interval 1 day
'''

        Con = pymysql.Connect(host="192.168.1.182", user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
              charset='utf8')

        robot_log = pd.read_sql_query(sql, Con)
        robot_log['id'] = robot_log['uniqueid'].apply(lambda x: x[:x.find(".",x.find(".",2)+1)])
        robot_log['new_name'] = robot_log.apply(lambda row: 'mtsby_ivi_1_'+str(row['phone'])+'_'+str(row['call_date'].strftime("%d%m%Y%H%M%S"))+'.mp3', axis=1)
        dialogs = robot_log['id'].to_list()


        call_date = str(robot_log['call_date'][0].strftime("%d-%m-%Y"))
        queue = 'just-call-9166-new/'
        audios = []
        name_audios = []
        for server in servers:
            url = 'http://'+server+f':9090/{call_date}/'+queue
    
            try:
                response = requests.get(url)
                files = response.text
        
                for i in files.split('\n'):
                    if "dialog.opus" in i:

                        audio_file = i[i.find('[SND]"></td><td><a href="')+25 : i.find('dialog.opus')+11]
                        ful_url = url+'/'+audio_file
                
                        if audio_file[:audio_file.find("_")] in dialogs:
    
                            audios.append(ful_url)
                            name_audios.append(audio_file[:audio_file.find("_")])
                    
            except :
                pass
        links = pd.DataFrame({'id': name_audios, 'links': audios})
        full = robot_log.merge(links, how='left', on='id')
        full = full[['links','new_name']].fillna('').query('links != ""').values.tolist()
        for row in full:
            url = row[0]
            filename = row[1]
    
            response = requests.get(url)
            with open(path + ('\\') + filename, 'wb') as file:
                file.write(response.content)
        import shutil
        audio_folder = 'C:/Users/Guest/Desktop/audios'

        files = os.listdir(audio_folder)

        max_size = 19 * 1024 * 1024

        current_size = 0
        current_zip_num = 1
        in_current_zip = []

        for file in files:
            file_path = os.path.join(audio_folder, file)
            file_size = os.path.getsize(file_path)
    
            if current_size + file_size > max_size:
                # Создаем новый архив
                with zipfile.ZipFile(f'архив_{current_zip_num}.zip', 'w') as zipf:
                    for audio_file in in_current_zip:
                        zipf.write(audio_file)
                
                current_zip_num += 1
                current_size = 0
                in_current_zip = []
    
            in_current_zip.append(file_path)
            current_size += file_size

        # Записываем оставшиеся файлы
        with zipfile.ZipFile(f'архив_{current_zip_num}.zip', 'w') as zipf:
            for audio_file in in_current_zip:
                zipf.write(audio_file)

                        
        for file in os.listdir(path):
            file_path = os.path.join(path, file)
            try:
                if os.path.isfile(file_path):
                    os.unlink(file_path)
            except Exception as e:
                print(f"Ошибка удаления файла {file}: {e}")
        directory = r'C:\Users\Guest\Pithon scrips'
        files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f)) and f.startswith('архив_') and f.endswith('.zip')]

        for file_name in files:
            with open(os.path.join(directory, file_name), 'rb') as file:
                await bot.send_document(chat_id, file)
        files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f)) and f.startswith('архив_')]
        for file_name in files:
            os.remove(os.path.join(directory, file_name))
                      
    except Exception as e:
        print(f"Ошибка: {e}")
        

In [9]:
@dp.message_handler(commands=['otchet_9066'])
async def otchet(message):
    chat_id = str(message.chat.id)
    await bot.send_message(chat_id, "Скрипт по отчету очереди 9066")
    
    import pandas as pd
    import pymysql
    import gspread 
    from oauth2client.service_account import ServiceAccountCredentials
    import datetime
    import os
    from datetime import datetime, timedelta
    import glob
    from datetime import datetime
    from clickhouse_driver import Client
    import re
    
    
    import pymysql.cursors
    import glob

    queue_list = [9066]

    user='robot_read_only'
    password='du9Itg5bnzTb'


    servers = ['192.168.1.15',
# '192.168.1.131',
'192.168.1.81',
'192.168.1.36',
'192.168.1.84',
'192.168.1.85',
# '192.168.1.123',
'192.168.1.103',
'192.168.1.86',
'192.168.1.122',
'192.168.1.124',
'192.168.1.125',
'192.168.1.127',
'192.168.1.129',
'192.168.1.126',
'192.168.1.130',
'192.168.1.87',
'192.168.1.59',
'192.168.1.114',
'192.168.1.146',
'192.168.1.109',
'192.168.1.107',
'192.168.1.156',
'192.168.1.110',
'192.168.1.113',
'192.168.1.118',
'192.168.1.116',
'192.168.1.147']
              


    no_data_rollback = pd.DataFrame()
    k = 0

    df_full = pd.DataFrame()

    for queue in queue_list:
        for server in servers:
            try:
                connection = pymysql.connect(host= server,
                                 user=user,
                                 password=password,
                                 database='robot',
                                 cursorclass=pymysql.cursors.DictCursor)
                with connection.cursor() as cursor:
        
                    sql = f'''select distinct phone,date,step_start,a.uniqueid uniq_cdr,normalized,full_normalized,`set`,`match`,result,{queue} as dialog
                 from `cel_just-call-{queue}-new_1`  a
                 left join `bill_cdr` b  on a.uniqueid = b.uniqueid
                 where # step_start in (3,4,5,6,7,260,261,262,92)
                 # and 
                 #(date(date) = DATE(now()) or
                 (date(date) = DATE(now())
                 or
                 date(date) = DATE(now())- interval 1 day)
                 '''
                    cursor.execute(sql)
                    result = cursor.fetchone()
                    df = pd.read_sql_query(sql, connection)
    
                    df_full = df_full.append(df)
            except Exception as e:
                print(f"Ошибка при подключении к серверу {server}: {e}")
                continue
                k += 1
        
        
    df_full2 = df_full[['phone','step_start','uniq_cdr','full_normalized']]
    df_full2['full_normalized'] = df_full2['full_normalized'].fillna('').apply(lambda x:  'тишина' if x == '' else x)
    df_full2['route'] = df_full2['step_start'].astype('str')+', абонент: '+df_full2['full_normalized'].fillna('')
    df_full2 = df_full2[['phone','route']].drop_duplicates()
    df_full2 = df_full2.groupby('phone')['route'].apply(lambda x: ', '.join(x)).reset_index()

    import pymysql

    billsec_full = pd.DataFrame()

    for queue in queue_list:
        sql = f'''select phone, real_billsec, billsec, last_step, call_date + interval 3 hour call_date,uniqueid,
    {queue} dialog
from suitecrm_robot.jc_robot_log
         left join suitecrm.contacts
                   on phone = phone_work
         left join suitecrm.contacts_cstm on contacts.id = id_c
where substring(dialog, 11, 4) = {queue}
  and (date(call_date) = date(now())
    or date(call_date) = date(now()) - interval 1 day)  and last_step in (91,93,95,97)
    '''

        Con = pymysql.Connect(host="192.168.1.182", user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
                      charset='utf8')

        billsec = pd.read_sql_query(sql, Con)
        billsec_full = billsec_full.append(billsec)
        k += 1
        
    try:    
        billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['real_billsec'] = billsec_full['real_billsec'].astype('str').apply(lambda x: x.replace('nan',''))
        billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('.0',''))
        billsec_full['billsec'] = billsec_full['billsec'].astype('str').apply(lambda x: x.replace('nan',''))    
        
        billsec_full = billsec_full.astype('str').fillna('')
        df_full2 = df_full2.astype('str')
        def check_bill(row):
            if pd.isnull(row['real_billsec']) or row['real_billsec'] == '' or row['real_billsec'] == 'None':
                return row['billsec']
            else:
                return row['real_billsec']
        
        billsec_full['real_billsec'] = billsec_full.apply(check_bill, axis=1)

        
        tab = billsec_full.merge(df_full2, left_on = 'phone', right_on ='phone', how = 'left').fillna('')
        tab=tab.astype('str')
        tab['Тип']=''
        tab['Лид']=''

        def update_step(row):
            if row['last_step'] == '91':
                row['Тип'] = 'Покупка'
                row['Лид'] = '91 ЛИД ПОКУПАЮ'
            elif row['last_step'] == '93':
                row['Тип'] = 'Покупка'
                row['Лид'] = '93 ЛИД ВОПРОС ПОКУПАЮ'
            elif row['last_step'] == '95':
                row['Тип'] = 'Продажа'
                row['Лид'] = '95 ЛИД ПРОДАЮ'
            elif row['last_step'] == '97':
                row['Тип'] = 'Продажа'
                row['Лид'] = '97 ЛИД ВОПРОС ПРОДАЮ'
            else:
                return row
        tab.apply(lambda row: update_step(row), axis=1)

#         tab = tab[['phone','start_time','report_status','duration','full_text']]
        tab.to_excel('Отчет по 9066.xlsx', index=False)
    
    # тут пишешь все свои действия по выгрузке, и потом отправляешь файл
        with open('Отчет по 9066.xlsx', 'rb') as file:
            await bot.send_document(chat_id, file)
    except Exception as e:
        print(f"Ошибка: {e}")
    

In [10]:
@dp.message_handler(content_types=['text'])
async def get_message(message):
    import datetime
    yesterday = datetime.date.today()
    answer = str(message.text) +" "+ str(yesterday.strftime("%d-%m-%Y"))
    print(answer) 

In [ ]:
nest_asyncio.apply()

if __name__ == "__main__":
    executor.start_polling(dp, skip_updates=True)

Updates were skipped successfully.
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:86: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:88: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_full = df_full.append(df)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:86: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:88: FutureWarning: The frame.append method is d

C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:86: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:88: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_full = df_full.append(df)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:86: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:88: FutureWarning: The frame.append method is deprecated and will be removed from 

C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1647262202.py:114: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1647262202.py:116: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_full = df_full.append(df)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1647262202.py:114: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1647262202.py:116: FutureWarning: The frame.append method is deprecated and will be removed f

C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1647262202.py:114: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1647262202.py:116: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_full = df_full.append(df)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1647262202.py:114: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1647262202.py:116: FutureWarning: The frame.append method is deprecated and will be removed f

C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:88: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_full = df_full.append(df)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:86: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:88: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_full = df_full.append(df)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\1847123926.py:86: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are no

...

C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:133: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:135: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_full = df_full.append(df)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:133: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:135: FutureWarning: The frame.append method is deprecated and will be removed f

C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:135: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_full = df_full.append(df)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:133: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:135: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_full = df_full.append(df)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:133: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects ar

C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:133: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:135: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_full = df_full.append(df)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:133: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:135: FutureWarning: The frame.append method is deprecated and will be removed f

C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:133: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:135: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_full = df_full.append(df)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:133: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, connection)
C:\Users\Guest\AppData\Local\Temp\ipykernel_21296\2808665025.py:135: FutureWarning: The frame.append method is deprecated and will be removed f

Cause exception while getting updates.
Traceback (most recent call last):
  File "C:\Users\Guest\anaconda3\lib\site-packages\aiogram\bot\api.py", line 139, in make_request
    async with session.post(url, data=req, **kwargs) as response:
  File "C:\Users\Guest\anaconda3\lib\site-packages\aiohttp\client.py", line 1167, in __aenter__
    self._resp = await self._coro
  File "C:\Users\Guest\anaconda3\lib\site-packages\aiohttp\client.py", line 586, in _request
    await resp.start(conn)
  File "C:\Users\Guest\anaconda3\lib\site-packages\aiohttp\client_reqrep.py", line 905, in start
    message, payload = await protocol.read()  # type: ignore[union-attr]
  File "C:\Users\Guest\anaconda3\lib\site-packages\aiohttp\streams.py", line 616, in read
    await self._waiter
  File "C:\Users\Guest\anaconda3\lib\asyncio\futures.py", line 285, in __await__
    yield self  # This tells Task to wait for completion.
  File "C:\Users\Guest\anaconda3\lib\asyncio\tasks.py", line 304, in __wakeup
    future.r